# RAG vs CAG

Dos formas distintas de darle contexto a un LLM:

| Patrón | Cómo |
|---|---|
| **RAG** · Retrieval-Augmented Generation | Recupera dinámicamente N chunks de un vectorstore por query |
| **CAG** · Cache-Augmented Generation | Mete el corpus **entero y estable** dentro del prompt y reutiliza su procesamiento entre preguntas |

**El plot twist 2026**: el CAG académico ("meter contexto a pelo") es un strawman. El CAG **real** y productivo en 2026 es **prompt caching nativo del proveedor** · OpenAI, Anthropic y Gemini lo soportan.

Este notebook hace dos pasadas:

1. **Pasada histórica** · CAG académico con `flan-t5-base` · para entender el concepto
2. **Pasada moderna** · RAG frente a CAG con prompt caching de OpenAI (`gpt-6-luna`) · el patrón real de 2026, con tokens, latencia, coste y calidad medidos

---

# Parte 1 · CAG académico (histórico, 2023)

Como referencia conceptual. **Esto NO es lo que harías en producción 2026**, pero ayuda a ver de dónde viene el concepto.

Comparamos sobre `flan-t5-base` (250M params, 2022):

- **RAG**: recuperar 2 chunks de un FAISS por query y meter en prompt
- **CAG**: usar un contexto fijo predefinido

In [1]:
%pip install -q transformers sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time, numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("  ✓ modelos cargados")

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  ✓ modelos cargados


### Corpus mini · 5 docs sobre RAG/CAG

In [3]:
documents = [
    "Retrieval-Augmented Generation (RAG) combines language models with external document retrieval to improve factual accuracy.",
    "Context-Augmented Generation (CAG) relies on supplying relevant background info in the prompt.",
    "FAISS enables fast similarity search across large text embeddings.",
    "Retrieval helps language models by grounding answers in actual data, reducing hallucination.",
    "Static context in prompts allows models to focus on specific themes but lacks flexibility.",
]

doc_embeddings = embedder.encode(documents, convert_to_numpy=True)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

query = "How does retrieval help improve generation?"
query_embedding = embedder.encode([query])

### RAG · recupera dinámicamente y genera

In [4]:
rag_start = time.time()
_, indices = index.search(query_embedding, k=2)
retrieved_docs = "\n".join(documents[i] for i in indices[0])

rag_prompt = f"""You are a helpful assistant.
Use the following context to answer the question concisely and clearly.
Context:
{retrieved_docs}
Question: {query}
Answer:"""

rag_inputs = tokenizer(rag_prompt, return_tensors="pt", truncation=True, max_length=512)
rag_output = model.generate(**rag_inputs, max_new_tokens=100)
rag_response = tokenizer.decode(rag_output[0], skip_special_tokens=True)
rag_time = time.time() - rag_start

rag_prompt_tokens = len(tokenizer.tokenize(rag_prompt))
rag_output_tokens = len(tokenizer.tokenize(rag_response))

### CAG · contexto fijo en el prompt

In [5]:
cag_start = time.time()

cag_context = (
    "Retrieval gives language models access to external knowledge, helping them "
    "reduce hallucination and improve factual accuracy during response generation."
)

cag_prompt = f"""You are a helpful assistant.
Use the provided context to answer the question.
Context:
{cag_context}
Question: {query}
Answer:"""

cag_inputs = tokenizer(cag_prompt, return_tensors="pt", truncation=True, max_length=512)
cag_output = model.generate(**cag_inputs, max_new_tokens=100)
cag_response = tokenizer.decode(cag_output[0], skip_special_tokens=True)
cag_time = time.time() - cag_start

cag_prompt_tokens = len(tokenizer.tokenize(cag_prompt))
cag_output_tokens = len(tokenizer.tokenize(cag_response))

### Comparativa

In [6]:
print(f"Query: {query}\n")
print(f"=== RAG ===")
print(f"  prompt tokens   : {rag_prompt_tokens}")
print(f"  output tokens   : {rag_output_tokens}")
print(f"  latencia        : {rag_time:.3f}s")
print(f"  respuesta       : {rag_response}")

print(f"\n=== CAG ===")
print(f"  prompt tokens   : {cag_prompt_tokens}")
print(f"  output tokens   : {cag_output_tokens}")
print(f"  latencia        : {cag_time:.3f}s")
print(f"  respuesta       : {cag_response}")

Query: How does retrieval help improve generation?

=== RAG ===
  prompt tokens   : 86
  output tokens   : 6
  latencia        : 1.957s
  respuesta       : grounding answers in actual data

=== CAG ===
  prompt tokens   : 59
  output tokens   : 7
  latencia        : 0.119s
  respuesta       : gives language models access to external knowledge


**Lectura del experimento (con asteriscos)**:

- El CAG "sale rápido" · pero estamos haciendo trampa · el contexto que metimos a mano contiene casi literal la respuesta
- El RAG "sale más lento" · pero porque incluye la búsqueda y mete contexto extra
- En un caso de uso real con un corpus de 10k docs, el CAG no escala (¿qué contexto metes?) y el RAG sí

El experimento ilustra el trade-off conceptual, **no es una guía productiva**. Para lo productivo, sigue leyendo.

---

# Parte 2 · CAG real en 2026 · prompt caching nativo

El CAG productivo en 2026 ya no se hace metiendo contexto en el prompt y pagándolo entero en cada llamada. Se hace con **prompt caching del proveedor**:

| Proveedor | Mecanismo | Duración | Precio del input cacheado |
|---|---|---|---|
| **OpenAI** | Automático: si el prompt empieza por un prefijo **idéntico** de al menos 1.024 tokens | al menos 30 min | lectura 0,10× · escritura 1,25× (desde GPT-5.6) |
| **Anthropic** | Explícito: marcas `cache_control` en los bloques que quieres cachear | 5 min (ampliable a 1 h) | lectura 0,1× · escritura más cara que el input normal |
| **Gemini** | `caches.create()` explícito (y caché implícita en modelos recientes) | configurable | depende del modelo |

**Caso de uso ideal**: tienes un manual / política / repo / contrato grande, **estable**, y muchas preguntas cortas distintas sobre él. Lo cacheas una vez y lo reutilizas mil veces.

En esta parte usamos **OpenAI con `gpt-6-luna`** (el modelo barato de la familia GPT-6) y comparamos las dos estrategias **sobre el mismo manual y las mismas preguntas**:

- **RAG**: trocear el manual, indexarlo con embeddings y meter en el prompt solo los 3 trozos más parecidos a cada pregunta.
- **CAG**: meter el manual **entero** al principio del prompt (parte estable) y la pregunta al final (parte variable). La primera llamada escribe el prefijo en caché; las siguientes lo leen a 0,10×.

Medimos de verdad, llamada a llamada: tokens de prompt, `cached_tokens`, latencia, coste en dólares y calidad de la respuesta.

In [7]:
%pip install -q "openai>=1.50" pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os, getpass, time, uuid, unicodedata
import numpy as np
import pandas as pd
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

client = OpenAI()

MODEL = "gpt-6-luna"                    # razonador barato de la familia GPT-6
EMBED_MODEL = "text-embedding-3-small"  # embeddings para el índice del RAG

# USD por millón de tokens (octubre 2026)
PRICES = {
    "gpt-6-luna": {"input": 0.10, "cached": 0.01, "write": 0.125, "output": 0.50},
    "gpt-6-sol":  {"input": 2.00, "cached": 0.20, "write": 2.50,  "output": 10.00},
}


def coste(usage, model=MODEL):
    """Coste real de una llamada: input normal + leído de caché + escrito en caché + output."""
    p = PRICES[model]
    d = usage.prompt_tokens_details
    leidos = (d.cached_tokens or 0) if d else 0
    # La API devuelve cache_write_tokens (tokens escritos en caché, a 1,25x); la versión del SDK
    # puede no declararlo como atributo, por eso se lee con getattr.
    escritos = (getattr(d, "cache_write_tokens", 0) or 0) if d else 0
    normal = usage.prompt_tokens - leidos - escritos
    return (normal * p["input"] + leidos * p["cached"] + escritos * p["write"]
            + usage.completion_tokens * p["output"]) / 1e6


def coste_sin_cache(usage, model=MODEL):
    """Lo que habría costado la misma llamada si no existiera la caché."""
    p = PRICES[model]
    return (usage.prompt_tokens * p["input"] + usage.completion_tokens * p["output"]) / 1e6

print(f"✓ cliente OpenAI listo · modelo {MODEL}")

✓ cliente OpenAI listo · modelo gpt-6-luna


### El corpus · el manual de atención al cliente de ACME

Para que la comparación sea justa necesitamos un corpus con **información real repartida por secciones** (no texto de relleno): políticas de devolución, envíos, garantía, Premium, outlet... y un catálogo de productos con precios y pesos.

El prompt caching de OpenAI exige **al menos 1.024 tokens** de prefijo idéntico. Nuestro manual mide unos 3.770 tokens según la propia API (lo verás en `cached_tokens`): suficiente para que se cachee (en producción serían 50.000 o más).

`RUN_ID` solo sirve para que cada ejecución empiece con la caché vacía y se vea la escritura de la primera llamada.

In [9]:
RUN_ID = uuid.uuid4().hex[:8]

SECCIONES = {
    "Plazo de devolución": (
        "Tienes 30 días naturales desde la fecha de entrega para devolver cualquier producto. "
        "El plazo se cuenta desde el día en que recibes el pedido, no desde el día de la compra. "
        "Para iniciar una devolución entra en el área de cliente, abre el pedido y pulsa 'Devolver artículos'. "
        "Recibirás una etiqueta de envío y un código de devolución que debes indicar en el paquete."
    ),
    "Estado del producto al devolverlo": (
        "El producto debe devolverse en su estado original: sin usar, con todos sus accesorios, manuales y "
        "etiquetas, y dentro de su embalaje original. Si el embalaje original falta o está muy dañado, "
        "ACME puede descontar hasta un 20% del importe reembolsado. Los productos que muestren señales de uso "
        "o golpes se revisan en el almacén y el reembolso puede ser parcial."
    ),
    "Excepciones a la devolución": (
        "No se admiten devoluciones de productos perecederos (café en grano, cápsulas, alimentación) salvo "
        "defecto de fábrica. Tampoco se devuelve el software descargado ni las licencias digitales una vez "
        "activadas. Los auriculares intraurales desprecintados no se aceptan por motivos de higiene."
    ),
    "Productos personalizados": (
        "Los productos personalizados (grabados, colores a medida o kits configurados por el cliente) no "
        "admiten devolución por desistimiento. Solo se aceptan si llegan con un defecto o si no coinciden "
        "con la personalización pedida. En ese caso ACME rehace el producto sin coste."
    ),
    "Outlet y rebajas": (
        "Los artículos comprados en la sección Outlet tienen un plazo de devolución reducido de 15 días "
        "naturales desde la entrega. Esta reducción se aplica a todos los clientes, también a los Premium: "
        "la ampliación de plazo de Premium no se aplica al Outlet. Los artículos rebajados fuera del Outlet "
        "mantienen el plazo normal."
    ),
    "Reembolsos": (
        "El reembolso se hace en el mismo método de pago usado en la compra, en un máximo de 14 días desde "
        "que el producto llega al almacén. Si pagaste con Bizum, el reembolso llega a la cuenta asociada al "
        "teléfono. Los gastos de envío originales solo se reembolsan si devuelves el pedido completo."
    ),
    "Envíos": (
        "Los pedidos se preparan en 24 horas laborables desde el almacén de Zaragoza. El envío estándar a la "
        "Península cuesta 4,95 euros y es gratis a partir de 50 euros. El envío exprés (entrega al día "
        "siguiente) cuesta 9,95 euros. Baleares: 9,95 euros y entrega en 3 a 5 días laborables."
    ),
    "Envíos internacionales": (
        "ACME envía a Portugal, Francia, Italia y Alemania con una tarifa única de 14,90 euros. No se hacen "
        "envíos fuera de la Unión Europea. Canarias, Ceuta y Melilla tienen tarifa propia de 19,90 euros y "
        "pueden aplicarse impuestos de importación que paga el cliente."
    ),
    "Garantía": (
        "Todos los productos tienen 3 años de garantía legal. Para tramitarla necesitas el número de pedido "
        "y una foto o vídeo del defecto. Si el producto no se puede reparar, se sustituye por uno igual o "
        "equivalente. Los daños por mal uso, caídas, humedad o cal acumulada no están cubiertos."
    ),
    "Suscripción Premium": (
        "Premium cuesta 39 euros al año e incluye envío estándar gratis sin mínimo, devoluciones ampliadas "
        "a 60 días (excepto en el Outlet) y acceso anticipado a las rebajas. Se renueva automáticamente cada "
        "año y se puede cancelar desde el área de cliente; no se reembolsa la parte proporcional."
    ),
    "Pagos y facturas": (
        "Aceptamos tarjeta, Bizum, PayPal y transferencia. El pago a plazos sin intereses en 3 meses está "
        "disponible para pedidos de más de 150 euros. Las facturas se descargan desde el área de cliente y "
        "se pueden emitir a nombre de empresa indicando el NIF antes de finalizar la compra."
    ),
    "Atención al cliente": (
        "El chat atiende de lunes a viernes de 9:00 a 20:00 y los sábados de 10:00 a 14:00. El teléfono "
        "900 123 456 es gratuito. Las incidencias de transporte deben comunicarse en un máximo de 48 horas "
        "desde la entrega, con fotos del paquete."
    ),
}

# Catálogo: 120 productos con precio y peso (datos concretos que el modelo tiene que encontrar)
TIPOS = ["cafetera", "molinillo", "hervidor", "tostadora", "batidora", "aspiradora",
         "auriculares", "altavoz", "báscula", "lámpara", "freidora de aire", "plancha"]
NOMBRES = ["Brava", "Serena", "Nube", "Faro", "Delta", "Lince", "Otoño", "Brisa",
           "Ébano", "Coral", "Atlas", "Duna", "Rumbo", "Vela", "Prisma", "Roca"]
catalogo = []
for i in range(120):
    nombre, tipo = NOMBRES[i % 16], TIPOS[i % 12]
    modelo = f"ACME {nombre}" + ["", " Plus", " Pro", " Mini", " Max", " Eco", " Duo", " Smart"][i // 16]
    precio = 29 + (i * 37) % 260
    peso = f"{0.4 + ((i * 7) % 45) / 10:.1f}".replace(".", ",")
    catalogo.append(f"- {modelo} ({tipo}): {precio},00 euros · peso {peso} kg")

INSTRUCCIONES = (
    "Eres el asistente de atención al cliente de la tienda online ACME. "
    "Responde en español, en dos o tres frases como máximo, usando SOLO la información del manual. "
    "Si la respuesta no está en el manual, dilo claramente."
)

MANUAL = (
    f"MANUAL DE ATENCIÓN AL CLIENTE DE ACME (versión {RUN_ID})\n\n"
    + "\n\n".join(f"## {titulo}\n{texto}" for titulo, texto in SECCIONES.items())
    + "\n\n## Catálogo de productos\n" + "\n".join(catalogo)
)
print(f"Manual: {len(MANUAL):,} caracteres · {len(SECCIONES)} secciones + {len(catalogo)} productos")
print([l for l in catalogo if "Brava (" in l])

Manual: 10,642 caracteres · 12 secciones + 120 productos
['- ACME Brava (cafetera): 29,00 euros · peso 0,4 kg']


### Las preguntas

Las tres preguntas de siempre sobre devoluciones, más dos que ponen a prueba cada estrategia:

- una que exige **cruzar dos secciones** (Premium + Outlet), donde el RAG puede quedarse corto si no recupera las dos;
- una que busca un **dato concreto del catálogo** (precio y peso de un producto entre 120).

Cada pregunta lleva los **hechos clave** que debería contener una buena respuesta. Es un control de calidad sencillo y transparente (búsqueda de cadenas), no una evaluación completa: eso lo verás en el módulo de evaluación de RAG.

In [10]:
PREGUNTAS = [
    {"q": "¿Cuántos días tengo para devolver un producto?",
     "hechos": [["30 dias", "30 días"]]},
    {"q": "¿Hay excepciones a la política de devolución?",
     "hechos": [["perecedero"], ["software", "licencia"], ["personalizado"], ["outlet", "15 dias", "15 días"]]},
    {"q": "¿Qué dice del estado del producto al devolverlo?",
     "hechos": [["embalaje"], ["sin usar", "estado original"], ["20%", "20 %"]]},
    {"q": "Soy cliente Premium y compré un artículo del Outlet hace 40 días. ¿Puedo devolverlo?",
     "hechos": [["15 dias", "15 días"], ["outlet"]]},
    {"q": "¿Cuánto cuesta la cafetera ACME Brava y cuánto pesa?",
     "hechos": [["29"], ["0,4", "0.4", "400 g"]]},
]


def normaliza(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))


def hechos_cubiertos(respuesta, hechos):
    """Cuántos hechos clave aparecen en la respuesta (cada hecho admite varias formas)."""
    r = normaliza(respuesta)
    return sum(any(normaliza(alt) in r for alt in hecho) for hecho in hechos)

print(f"{len(PREGUNTAS)} preguntas listas")

5 preguntas listas


### RAG · indexar el manual y recuperar 3 trozos por pregunta

Troceamos por secciones (cada sección del manual es un trozo y el catálogo se parte en bloques de 12 productos), calculamos los embeddings con `text-embedding-3-small` y buscamos por similitud coseno con NumPy. Con 22 trozos no hace falta un vectorstore: el mecanismo es el mismo que viste con FAISS o Chroma.

In [11]:
trozos = [f"## {titulo}\n{texto}" for titulo, texto in SECCIONES.items()]
trozos += ["## Catálogo de productos\n" + "\n".join(catalogo[i:i + 12]) for i in range(0, len(catalogo), 12)]

emb = client.embeddings.create(model=EMBED_MODEL, input=trozos)
M = np.array([e.embedding for e in emb.data])
M /= np.linalg.norm(M, axis=1, keepdims=True)
print(f"{len(trozos)} trozos indexados · {emb.usage.total_tokens:,} tokens de embedding (se paga una sola vez)")


def recuperar(pregunta, k=3):
    q = client.embeddings.create(model=EMBED_MODEL, input=[pregunta]).data[0].embedding
    q = np.array(q) / np.linalg.norm(q)
    top = np.argsort(M @ q)[::-1][:k]
    return [trozos[i] for i in top]

for t in recuperar(PREGUNTAS[3]["q"]):
    print("·", t.splitlines()[0])

22 trozos indexados · 3,986 tokens de embedding (se paga una sola vez)
· ## Outlet y rebajas
· ## Plazo de devolución
· ## Suscripción Premium


### Dos estrategias, un mismo modelo

La única diferencia entre las dos funciones es **qué contexto va en el prompt y en qué orden**:

| | Mensaje `system` | Mensaje `user` |
|---|---|---|
| **RAG** | instrucciones | 3 trozos recuperados + pregunta |
| **CAG** | instrucciones + **manual entero** (estable, idéntico en todas las llamadas) | solo la pregunta (variable, al final) |

En la versión anterior de este notebook, con Anthropic, había que marcar el bloque grande con `cache_control`. En OpenAI no se marca nada: la caché se activa sola cuando el prompt empieza por el mismo prefijo de al menos 1.024 tokens, así que lo único que tienes que cuidar es **poner lo estable primero y lo variable al final**. `usage.prompt_tokens_details.cached_tokens` te dice cuántos tokens se han leído de caché.

GPT-6 son modelos razonadores: no aceptan `temperature` ni `max_tokens`. Usamos `max_completion_tokens` y `reasoning_effort="none"` (preguntas de consulta directa, no hace falta pensar).

In [12]:
SYSTEM_CAG = f"{INSTRUCCIONES}\n\n{MANUAL}"


def llamar(system, user, estrategia, pregunta, t_retrieval=0.0, trozos_usados="(manual entero)"):
    t0 = time.time()
    r = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        max_completion_tokens=400,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    t_llm = time.time() - t0
    u, d = r.usage, r.usage.prompt_tokens_details
    respuesta = r.choices[0].message.content or ""
    return {
        "estrategia": estrategia,
        "pregunta": pregunta["q"][:48],
        "prompt_tokens": u.prompt_tokens,
        "cached_tokens": (d.cached_tokens or 0) if d else 0,
        "escritos_cache": (getattr(d, "cache_write_tokens", 0) or 0) if d else 0,
        "output_tokens": u.completion_tokens,
        "latencia_s": round(t_retrieval + t_llm, 2),
        "coste_$": coste(u),
        "coste_sin_cache_$": coste_sin_cache(u),
        "hechos": f"{hechos_cubiertos(respuesta, pregunta['hechos'])}/{len(pregunta['hechos'])}",
        "trozos": trozos_usados,
        "respuesta": respuesta,
    }


def preguntar_rag(pregunta, k=3):
    t0 = time.time()
    recuperados = recuperar(pregunta["q"], k)
    t_retrieval = time.time() - t0
    contexto = "\n\n".join(recuperados)
    user = f"Fragmentos del manual:\n\n{contexto}\n\nPregunta: {pregunta['q']}"
    titulos = " | ".join(t.splitlines()[0].removeprefix("## ") for t in recuperados)
    return llamar(INSTRUCCIONES, user, "RAG", pregunta, t_retrieval, titulos)


def preguntar_cag(pregunta):
    return llamar(SYSTEM_CAG, pregunta["q"], "CAG", pregunta)

### Ronda 1 · RAG

Cada pregunta paga solo los trozos recuperados. Son prompts pequeños (por debajo de 1.024 tokens) y distintos en cada llamada, así que aquí la caché no entra en juego. La latencia incluye la búsqueda (embedding de la pregunta + similitud).

In [13]:
filas_rag = [preguntar_rag(p) for p in PREGUNTAS]
df_rag = pd.DataFrame(filas_rag)
df_rag.drop(columns=["respuesta", "estrategia"]).style.format({"latencia_s": "{:.2f}", "coste_$": "{:.6f}", "coste_sin_cache_$": "{:.6f}"})

,pregunta,prompt_tokens,cached_tokens,escritos_cache,output_tokens,latencia_s,coste_$,coste_sin_cache_$,hechos,trozos
0,¿Cuántos días tengo para devolver un producto?,330,0,0,18,0.94,0.000042,0.000042,1/1,Plazo de devolución | Estado del producto al devolverlo | Reembolsos
1,¿Hay excepciones a la política de devolución?,305,0,0,78,1.36,0.000069,0.000069,3/4,Excepciones a la devolución | Estado del producto al devolverlo | Productos personalizados
2,¿Qué dice del estado del producto al devolverlo?,306,0,0,68,1.24,0.000065,0.000065,3/3,Estado del producto al devolverlo | Productos personalizados | Excepciones a la devolución
3,Soy cliente Premium y compré un artículo del Out,308,0,0,28,1.07,0.000045,0.000045,2/2,Outlet y rebajas | Plazo de devolución | Suscripción Premium
4,¿Cuánto cuesta la cafetera ACME Brava y cuánto p,929,0,0,24,0.86,0.000105,0.000105,2/2,Catálogo de productos | Catálogo de productos | Catálogo de productos


### Ronda 2 · CAG con prompt caching

Mismas preguntas, en el mismo orden. La **1ª llamada** manda el manual entero por primera vez y lo escribe en caché; de la **2ª en adelante** el prefijo coincide y se lee de caché.

In [14]:
filas_cag = [preguntar_cag(p) for p in PREGUNTAS]
df_cag = pd.DataFrame(filas_cag)
df_cag.drop(columns=["respuesta", "estrategia", "trozos"]).style.format({"latencia_s": "{:.2f}", "coste_$": "{:.6f}", "coste_sin_cache_$": "{:.6f}"})

,pregunta,prompt_tokens,cached_tokens,escritos_cache,output_tokens,latencia_s,coste_$,coste_sin_cache_$,hechos
0,¿Cuántos días tengo para devolver un producto?,3782,0,3779,37,1.98,0.000491,0.000397,1/1
1,¿Hay excepciones a la política de devolución?,3782,3765,14,70,8.74,0.000075,0.000413,3/4
2,¿Qué dice del estado del producto al devolverlo?,3782,3765,14,68,1.16,0.000074,0.000412,3/3
3,Soy cliente Premium y compré un artículo del Out,3793,3765,25,34,1.12,0.000058,0.000396,2/2
4,¿Cuánto cuesta la cafetera ACME Brava y cuánto p,3786,3765,18,24,1.07,0.000052,0.000391,2/2


### Comparativa · tokens, latencia y coste por pregunta

In [15]:
def resumen(df):
    return {
        "prompt_tokens_medio": round(df.prompt_tokens.mean()),
        "cached_tokens_total": int(df.cached_tokens.sum()),
        "coste_1a_llamada_$": df["coste_$"].iloc[0],
        "coste_medio_siguientes_$": df["coste_$"].iloc[1:].mean(),
        "coste_total_$": df["coste_$"].sum(),
        "latencia_1a_s": df.latencia_s.iloc[0],
        "latencia_media_siguientes_s": round(df.latencia_s.iloc[1:].mean(), 2),
        "hechos_cubiertos": f"{sum(int(h.split('/')[0]) for h in df.hechos)}/{sum(int(h.split('/')[1]) for h in df.hechos)}",
    }

tabla = pd.DataFrame({"RAG (k=3)": resumen(df_rag), "CAG (caché)": resumen(df_cag)})
cag_sin_cache = df_cag["coste_sin_cache_$"].sum()
print(f"CAG sin caché habría costado {cag_sin_cache:.6f} $ en total (con caché: {df_cag['coste_$'].sum():.6f} $)")
tabla

CAG sin caché habría costado 0.002009 $ en total (con caché: 0.000750 $)


,RAG (k=3),CAG (caché)
prompt_tokens_medio,436,3785
cached_tokens_total,0,15060
coste_1a_llamada_$,0.000042,0.000491
coste_medio_siguientes_$,0.000071,0.000065
coste_total_$,0.000326,0.00075
latencia_1a_s,0.94,1.98
latencia_media_siguientes_s,1.13,3.02
hechos_cubiertos,11/12,11/12


**Lectura de la ejecución guardada** (14/10/2026, `gpt-6-luna`). Si lo ejecutas tú, las cifras cambiarán un poco, sobre todo la latencia.

| | RAG (k=3) | CAG con caché |
|---|---|---|
| Tokens de prompt por pregunta | 305 a 929 (media 436) | 3.782 a 3.793 |
| `cached_tokens` | 0 en todas | 0 en la 1ª · 3.765 en cada una de las siguientes |
| Coste de la 1ª pregunta | 0,000042 $ | 0,000491 $ (escribe 3.779 tokens a 1,25×) |
| Coste medio de las siguientes | 0,000071 $ | 0,000065 $ |
| Total de las 5 preguntas | 0,000326 $ | 0,000750 $ (sin caché habría sido 0,002009 $) |
| Latencia 1ª / media de las siguientes | 0,94 s / 1,13 s | 1,98 s / 3,02 s (con un pico de 8,74 s) |

- **La caché funciona como promete.** Desde la 2ª llamada, 3.765 de los ~3.785 tokens del prompt se leen de caché y cada pregunta CAG baja de unos 0,0004 $ (lo que costaría sin caché) a unos 0,00006 $: unas 6 veces menos. Frente a un CAG sin caché, compensa ya en la 2ª pregunta.
- **Escribir no es gratis.** La 1ª llamada CAG cuesta 0,000491 $, un 24% más que sin caché (0,000397 $): es el 1,25× de escritura. Las siguientes llamadas también escriben 14 a 25 tokens: la pregunta nueva se añade a la caché. La API lo devuelve en `usage.prompt_tokens_details.cache_write_tokens`.
- **Con la caché caliente, CAG y RAG cuestan casi lo mismo por pregunta** (0,000065 $ frente a 0,000071 $) porque este manual es pequeño. Pero en el total de 5 preguntas el RAG sale 2,3 veces más barato: con pocas preguntas, la escritura inicial pesa mucho.
- **Latencia: la caché no te la regala aquí.** El RAG tarda 0,86 a 1,36 s (búsqueda incluida). Las llamadas CAG con caché tardaron 1,07 a 1,16 s, salvo un pico de 8,74 s en la 2ª pregunta que dispara la media (por eso en producción se mira la mediana o el p95, no la media). En otras dos ejecuciones de prueba el CAG estuvo entre 1,7 y 5,6 s y el RAG entre 1,1 y 1,7 s. Con un prefijo de ~3.800 tokens la caché recorta sobre todo **coste**: la latencia es ruidosa y no mejora de forma clara entre la 1ª llamada y las siguientes. La mejora de latencia se nota con prefijos de decenas de miles de tokens, frente a mandar ese mismo prefijo sin caché (no frente a un RAG de 400 tokens).

### Calidad · ¿qué ha respondido cada estrategia?

In [16]:
for p_, r, c in zip(PREGUNTAS, filas_rag, filas_cag):
    print(f"❓ {p_['q']}")
    print(f"   RAG [{r['hechos']}] {r['respuesta']}")
    print(f"       (trozos recuperados: {r['trozos']})")
    print(f"   CAG [{c['hechos']}] {c['respuesta']}\n")

❓ ¿Cuántos días tengo para devolver un producto?
   RAG [1/1] Tienes 30 días naturales desde la fecha de entrega para devolverlo.
       (trozos recuperados: Plazo de devolución | Estado del producto al devolverlo | Reembolsos)
   CAG [1/1] Tienes 30 días naturales desde la fecha de entrega para devolver un producto. Si lo compraste en la sección Outlet, el plazo es de 15 días naturales.

❓ ¿Hay excepciones a la política de devolución?
   RAG [3/4] Sí. No se admiten devoluciones de productos perecederos salvo defecto de fábrica, software descargado o licencias digitales activadas, auriculares intraurales desprecintados ni productos personalizados por desistimiento. Los personalizados solo se aceptan si llegan defectuosos o no coinciden con lo solicitado; en ese caso, ACME los rehace sin coste.
       (trozos recuperados: Excepciones a la devolución | Estado del producto al devolverlo | Productos personalizados)
   CAG [3/4] Sí. No se admiten devoluciones de productos perecederos salvo 

**Lectura de la calidad**

- Las dos estrategias cubren **11 de 12 hechos clave**. En el estado del producto, la pregunta Premium + Outlet y la cafetera dan prácticamente la misma respuesta.
- El RAG ha recuperado bien: Outlet + Plazo + Premium para la pregunta cruzada y los bloques del catálogo para la cafetera (esa es la pregunta RAG más cara, 929 tokens, porque cada bloque del catálogo es grande).
- Donde se nota que el CAG ve el manual entero es en los **matices**: a "¿cuántos días tengo?" añade por su cuenta que en el Outlet el plazo es de 15 días. El RAG, que recuperó Plazo, Estado y Reembolsos, no puede mencionar lo que no ha recuperado.
- El hecho que falla es el mismo en las dos: ninguna cita el plazo reducido del Outlet como excepción. El RAG no recuperó esa sección (trajo Excepciones, Estado y Personalizados); el CAG la tenía delante y no la usó. **Tener todo el contexto no garantiza usarlo**: por eso hay que evaluar, que es lo que harás en el módulo de evaluación de RAG.

### ¿Hasta qué tamaño de corpus compensa el CAG?

Con la caché caliente, una pregunta CAG paga el manual entero a 0,10× del input; una pregunta RAG paga solo sus trozos a precio completo. Con los tokens medidos arriba podemos calcular a partir de qué tamaño de manual el CAG pasa a ser **más caro** que el RAG (solo input; el output es parecido en las dos).

In [17]:
p = PRICES[MODEL]
rag_input = df_rag.prompt_tokens.mean()                    # tokens medidos por pregunta RAG
pregunta_cag = (df_cag.prompt_tokens - df_cag.cached_tokens).iloc[1:].mean()  # parte no cacheada del CAG
# Esa parte no cacheada (la pregunta) también se escribe en caché, así que se paga a precio de escritura
equilibrio = (rag_input * p["input"] - pregunta_cag * p["write"]) / p["cached"]
print(f"RAG: {rag_input:.0f} tokens de input por pregunta · CAG caliente: {pregunta_cag:.0f} sin cachear + el manual cacheado")
print(f"Punto de equilibrio: un manual de ~{equilibrio:,.0f} tokens. Por debajo, el CAG caliente sale más barato por pregunta.\n")

for tam in [int(df_cag.cached_tokens.max()), 50_000, 200_000, 1_000_000]:
    cag = (tam * p["cached"] + pregunta_cag * p["write"]) / 1e6
    rag = rag_input * p["input"] / 1e6
    print(f"manual de {tam:>9,} tokens · input por pregunta  CAG caliente {cag:.6f} $  vs  RAG {rag:.6f} $")

RAG: 436 tokens de input por pregunta · CAG caliente: 21 sin cachear + el manual cacheado
Punto de equilibrio: un manual de ~4,097 tokens. Por debajo, el CAG caliente sale más barato por pregunta.

manual de     3,765 tokens · input por pregunta  CAG caliente 0.000040 $  vs  RAG 0.000044 $
manual de    50,000 tokens · input por pregunta  CAG caliente 0.000503 $  vs  RAG 0.000044 $
manual de   200,000 tokens · input por pregunta  CAG caliente 0.002003 $  vs  RAG 0.000044 $
manual de 1,000,000 tokens · input por pregunta  CAG caliente 0.010003 $  vs  RAG 0.000044 $


**Lectura del punto de equilibrio**: sale en unos 4.100 tokens. Nuestro manual (3.765 tokens cacheados) está justo por debajo, por eso el CAG caliente sale un poco más barato que el RAG (0,000040 $ frente a 0,000044 $ de input por pregunta). Con un manual de 50.000 tokens cada pregunta CAG costaría 0,000503 $ de input, unas 11 veces más que el RAG (0,000044 $), y con 1 millón de tokens, 0,010003 $, además de tener que caber en la ventana de contexto del modelo. **El RAG paga por lo que recupera; el CAG paga (aunque sea a 0,10×) por todo el corpus en cada pregunta.**

## Decision matrix · ¿RAG, CAG (prompt caching) o híbrido?

| Tu caso | Recomendación |
|---|---|
| Corpus grande o que cambia a menudo | **RAG** · vectorstore |
| Corpus estable y pequeño (unos pocos miles de tokens) con muchas preguntas | **CAG con prompt caching** · con la caché caliente cuesta lo mismo que el RAG y ve todo el contexto |
| Pocas preguntas sobre el mismo documento | **RAG** · la escritura en caché (1,25×) no llega a amortizarse |
| Preguntas que cruzan muchas secciones | **CAG**, o RAG con más `k` y reranking |
| Necesitas citar fuentes específicas | **RAG** · es más fácil trazar |
| Latencia crítica (chat en vivo) | **Mide** · en nuestras pruebas el RAG, con prompts de ~400 tokens, fue igual o más rápido que el CAG con caché |
| Escalable a millones de docs | **RAG** · el CAG no escala más allá de la ventana de contexto |

**Híbrido**: usa RAG para reducir el corpus a los trozos relevantes para una conversación, **luego cachea** ese subconjunto como prefijo estable durante toda la conversación. Lo mejor de ambos.

## Cierre

- El CAG académico es útil **conceptualmente**, no productivamente
- El CAG **2026** es prompt caching nativo · en OpenAI no cambias ni una línea: pones lo estable primero, lo variable al final, y compruebas `cached_tokens`
- La caché abarata el input repetido (0,10×), pero la primera escritura cuesta 1,25×: compensa cuando el mismo prefijo se reutiliza muchas veces
- RAG sigue siendo el patrón principal · CAG es la **optimización** cuando el contexto es estable y se reusa
- Patrón productivo más común: **RAG → cachea el contexto resultante para la conversación**

Más:

- [OpenAI prompt caching](https://platform.openai.com/docs/guides/prompt-caching)
- [Anthropic prompt caching docs](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)
- [Gemini context caching](https://ai.google.dev/gemini-api/docs/caching)